### PySpark MLlib
Lab Exercise Using PySpark and MLlib for K-means Clustering and Logistic Regression

In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
.appName("MLlib_Lab_Exercise") \
.config("spark.jars", "/Users/rattanak/jars/graphframes-0.8.4-spark3.5-s_2.12.jar") \
.getOrCreate()

# Verify Spark session
print(spark)

25/04/29 09:56:58 WARN Utils: Your hostname, Rattanaks-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 172.23.0.202 instead (on interface en0)
25/04/29 09:56:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/04/29 09:56:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Step 3: Load or Create a Sample Dataset
For this exercise, we’ll use the Iris dataset (available in many formats) or create a synthetic
dataset. Here, we’ll load the Iris dataset from a CSV file or use a sample dataset.
Load Iris Dataset Download the Iris dataset (e.g., from UCI Machine Learning Repository) and
save it as iris.csv. Example structure:

In [2]:
# Load Iris dataset
data = spark.read.csv("Iris.csv", header=True, inferSchema=True)

# Show first few rows
data.show(5)

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows



### Step 4: Data Preprocessing
For MLlib, features need to be combined into a single vector column.

In [3]:
from pyspark.ml.feature import VectorAssembler
# Define feature columns (adjust based on your dataset)
feature_columns = ["SepalLengthCm", "SepalWidthCm"] # Or ["sepal_length","sepal_width", "petal_length", "petal_width"]
# Create a VectorAssembler
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
# Transform data
data_with_features = assembler.transform(data)
data_with_features.show(5)

+---+-------------+------------+-------------+------------+-----------+---------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species| features|
+---+-------------+------------+-------------+------------+-----------+---------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|[5.1,3.5]|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|[4.9,3.0]|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|[4.7,3.2]|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|[4.6,3.1]|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|[5.0,3.6]|
+---+-------------+------------+-------------+------------+-----------+---------+
only showing top 5 rows



### Part 2: K-means Clustering
Step 5: Apply K-means Clustering
Use MLlib’s KMeans algorithm to cluster the data.

In [4]:
from pyspark.ml.clustering import KMeans

# Initialize K-means model
kmeans = KMeans().setK(3).setSeed(42).setFeaturesCol("features").setPredictionCol("cluster")

# Fit the model
kmeans_model = kmeans.fit(data_with_features)
# Make predictions
clustered_data = kmeans_model.transform(data_with_features)
# Show clustering results
clustered_data.select("features", "cluster").show(5)

25/04/29 09:57:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+---------+-------+
| features|cluster|
+---------+-------+
|[5.1,3.5]|      1|
|[4.9,3.0]|      1|
|[4.7,3.2]|      1|
|[4.6,3.1]|      1|
|[5.0,3.6]|      1|
+---------+-------+
only showing top 5 rows



### Step 6: Evaluate Clustering
Compute the Silhouette score to evaluate the clustering quality.

In [5]:
from pyspark.ml.evaluation import ClusteringEvaluator
# Evaluate clustering
evaluator = ClusteringEvaluator(predictionCol="cluster",
featuresCol="features", metricName="silhouette")
silhouette_score = evaluator.evaluate(clustered_data)
print(f"Silhouette Score: {silhouette_score}")

"""Interpretation
• A silhouette score close to 1 indicates well-separated clusters.
• A score near 0 indicates overlapping clusters.
• A negative score suggests poor clustering."""

Silhouette Score: 0.6179957264110181


'Interpretation\n• A silhouette score close to 1 indicates well-separated clusters.\n• A score near 0 indicates overlapping clusters.\n• A negative score suggests poor clustering.'

### Part 3: Logistic Regression
Step 7: Prepare Data for Classification
For Logistic Regression, we need a numeric label column. Convert the categorical label or species column to numeric using StringIndexer.

In [7]:
from pyspark.ml.feature import StringIndexer

# Convert categorical label to numeric
indexer = StringIndexer(inputCol="Species", outputCol="label_index") # Or" species" for Iris
indexed_data = indexer.fit(data_with_features).transform(data_with_features)
# Select features and label
final_data = indexed_data.select("features", "label_index")
final_data.show(5)

+---------+-----------+
| features|label_index|
+---------+-----------+
|[5.1,3.5]|        0.0|
|[4.9,3.0]|        0.0|
|[4.7,3.2]|        0.0|
|[4.6,3.1]|        0.0|
|[5.0,3.6]|        0.0|
+---------+-----------+
only showing top 5 rows



### Step 8: Split Data into Training and Test Sets
Split the data into 70% training and 30% testing.

In [8]:
# Split data
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)